In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
from functools import reduce

# Get the active session
session = get_active_session()

# ==========================================
# 1. WIN RATE FUNCTION
# ==========================================
def get_win_rate_data_all_metrics(quarter, region=None, segment=None):
    """
    Retrieves Win Rate data.
    - If 'segment' is provided: Filters specifically for that segment.
    - If 'segment' is None: Defaults to 'Enterprise/Commercial' (Excludes SMB/Digital).
    - If 'region' is None: Runs globally.
    """
    
    # Dynamic Filtering Logic
    region_filter = f"AND gtmi.region = '{region}'" if region else ""
    segment_filter = (
        f"AND gtmi.pro_forma_market_segment = '{segment}'" 
        if segment 
        else "AND gtmi.pro_forma_market_segment NOT IN ('SMB', 'Digital')"
    )
    
    label_region = region if region else "Global"
    label_segment = segment if segment else "Ent/Comm (Std)"

    query = f"""
    WITH part AS (
        SELECT id, partner, partner_deal_source
        FROM functional.gtm_sales_ops.partner_opp_table_all
    ),
    csql AS (
        SELECT id AS opportunity_id, campaign_id
        FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_BCV
    ),
    all_opps_consolidated_cte AS (
        SELECT
            gtmi.crm_opportunity_id,
            gtmi.product_arr_usd,
            
            -- STATUS
            CASE WHEN gtmi.opportunity_status = 'Closed' THEN 1 ELSE 0 END AS is_won,
            CASE WHEN gtmi.opportunity_status = 'Lost' AND (gtmi.deal_lost_reasonmulti__c NOT LIKE '%Duplicate%' OR gtmi.deal_lost_reasonmulti__c IS NULL) THEN 1 ELSE 0 END AS is_lost,

            -- SEGMENT FLAGS
            CASE WHEN gtmi.PRODUCT IN ('Contact_Center') THEN 1 ELSE 0 END AS is_CCaaS,
            CASE WHEN csql.campaign_id LIKE '%70180000001JlouAAC%' THEN 1 ELSE 0 END AS is_CSQL,
            CASE WHEN (gtmi.PRODUCT IN ('ES') OR gtmi.use_case_c ILIKE '%internal%') AND gtmi.opportunity_type ILIKE '%Expansion%' THEN 1 ELSE 0 END AS is_ES_Cross_Sell,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%BDR%' THEN 1 ELSE 0 END AS is_New_Cust_BDR,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%AE Only%' THEN 1 ELSE 0 END AS is_New_Cust_AE,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Zendesk Sourced' THEN 1 ELSE 0 END AS is_Partner_Influenced,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Partner Sourced' THEN 1 ELSE 0 END AS is_Partner_Sourced,
            CASE WHEN gtmi.PRODUCT IN ('AI_Expert', 'AR', 'Copilot', 'Ultimate') THEN 1 ELSE 0 END AS is_AI_group

        FROM functional.gtm_sales_ops.gtmsi_consolidated_pipeline_bookings gtmi
        LEFT JOIN part ON part.id = gtmi.crm_opportunity_id
        LEFT JOIN csql ON csql.opportunity_id = gtmi.crm_opportunity_id
        WHERE gtmi.date_label = 'today'
          AND gtmi.opportunity_is_commissionable = true
          AND (gtmi.product_arr_usd > 0 OR gtmi.product_booking_arr_usd > 0)
          AND gtmi.close_year_quarter = '{quarter}'
          {region_filter}
          {segment_filter}
    )

    SELECT 
        '{label_region}' as "Region",
        '{quarter}' as "Quarter",
        '{label_segment}' as "Segment",
        
        -- METRICS (Condensed for brevity - same calculations as before)
        ROUND(SUM(CASE WHEN is_CCaaS=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CCaaS=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "CCaaS: Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_CCaaS=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_CCaaS=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "CCaaS: Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_CCaaS=1 AND is_won=1 THEN crm_opportunity_id END) as "CCaaS: Won Transactions",

        ROUND(SUM(CASE WHEN is_CSQL=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CSQL=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "CSQL: Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_CSQL=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_CSQL=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "CSQL: Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_CSQL=1 AND is_won=1 THEN crm_opportunity_id END) as "CSQL: Won Transactions",

        ROUND(SUM(CASE WHEN is_ES_Cross_Sell=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_ES_Cross_Sell=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "ES Cross-sell: Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_ES_Cross_Sell=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_ES_Cross_Sell=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "ES Cross-sell: Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_ES_Cross_Sell=1 AND is_won=1 THEN crm_opportunity_id END) as "ES Cross-sell: Won Transactions",

        ROUND(SUM(CASE WHEN is_New_Cust_BDR=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_BDR=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "New Customer (BDR): Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_New_Cust_BDR=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_New_Cust_BDR=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "New Customer (BDR): Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_New_Cust_BDR=1 AND is_won=1 THEN crm_opportunity_id END) as "New Customer (BDR): Won Transactions",

        ROUND(SUM(CASE WHEN is_New_Cust_AE=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_AE=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "New Customer (AE): Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_New_Cust_AE=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_New_Cust_AE=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "New Customer (AE): Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_New_Cust_AE=1 AND is_won=1 THEN crm_opportunity_id END) as "New Customer (AE): Won Transactions",

        ROUND(SUM(CASE WHEN is_Partner_Influenced=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Influenced=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "Partner (Influenced): Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_Partner_Influenced=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_Partner_Influenced=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "Partner (Influenced): Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_Partner_Influenced=1 AND is_won=1 THEN crm_opportunity_id END) as "Partner (Influenced): Won Transactions",

        ROUND(SUM(CASE WHEN is_Partner_Sourced=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Sourced=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "Partner (Sourced): Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_Partner_Sourced=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_Partner_Sourced=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "Partner (Sourced): Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_Partner_Sourced=1 AND is_won=1 THEN crm_opportunity_id END) as "Partner (Sourced): Won Transactions",

        ROUND(SUM(CASE WHEN is_AI_group=1 AND is_won=1 THEN product_arr_usd ELSE 0 END) / NULLIF(SUM(CASE WHEN is_AI_group=1 AND (is_won=1 OR is_lost=1) THEN product_arr_usd ELSE 0 END),0) * 100, 2) as "AI: Win Rate Revenue ($)",
        ROUND(COUNT(DISTINCT CASE WHEN is_AI_group=1 AND is_won=1 THEN crm_opportunity_id END) / NULLIF(COUNT(DISTINCT CASE WHEN is_AI_group=1 AND (is_won=1 OR is_lost=1) THEN crm_opportunity_id END),0) * 100.0, 2) as "AI: Win Rate Transaction (#)",
        COUNT(DISTINCT CASE WHEN is_AI_group=1 AND is_won=1 THEN crm_opportunity_id END) as "AI: Won Transactions"
    FROM all_opps_consolidated_cte
    GROUP BY 1, 2, 3
    """
    return session.sql(query).to_pandas()


# ==========================================
# 2. SLIPPAGE FUNCTION
# ==========================================
def get_slippage_data_all_metrics(quarter, region=None, segment=None):
    
    region_filter = f"AND gtmi.region = '{region}'" if region else ""
    segment_filter = (
        f"AND gtmi.pro_forma_market_segment = '{segment}'" 
        if segment 
        else "AND gtmi.pro_forma_market_segment NOT IN ('SMB', 'Digital')"
    )
    
    label_region = region if region else "Global"
    label_segment = segment if segment else "Ent/Comm (Std)"

    query = f"""
    WITH initial_opp_stage AS (
        SELECT DISTINCT source_snapshot_date as initial_snapshot, crm_opportunity_id, opportunity_stage_name, OPPORTUNITY_CLOSE_DATE as initial_closedate
        FROM foundational.customer.dim_crm_opportunities_daily_snapshot
        WHERE source_snapshot_date = (select dateadd(day, -28, max(source_snapshot_date)) from foundational.customer.dim_crm_opportunities_daily_snapshot)
    ),
    csql AS (SELECT id AS opportunity_id, campaign_id FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_BCV),
    part AS (SELECT id, partner, partner_deal_source FROM functional.gtm_sales_ops.partner_opp_table_all),
    current_state AS (
        SELECT gtmi.crm_opportunity_id, gtmi.closedate as current_closedate,
            CASE WHEN gtmi.PRODUCT IN ('Contact_Center') THEN 1 ELSE 0 END AS is_CCaaS,
            CASE WHEN csql.campaign_id LIKE '%70180000001JlouAAC%' THEN 1 ELSE 0 END AS is_CSQL,
            CASE WHEN (gtmi.PRODUCT IN ('ES') OR gtmi.use_case_c ILIKE '%internal%') AND gtmi.opportunity_type ILIKE '%Expansion%' THEN 1 ELSE 0 END AS is_ES_Cross_Sell,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%BDR%' THEN 1 ELSE 0 END AS is_New_Cust_BDR,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%AE Only%' THEN 1 ELSE 0 END AS is_New_Cust_AE,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Zendesk Sourced' THEN 1 ELSE 0 END AS is_Partner_Influenced,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Partner Sourced' THEN 1 ELSE 0 END AS is_Partner_Sourced,
            CASE WHEN gtmi.PRODUCT IN ('AI_Expert', 'AR', 'Copilot', 'Ultimate') THEN 1 ELSE 0 END AS is_AI_group
        FROM functional.gtm_sales_ops.gtmsi_consolidated_pipeline_bookings gtmi
        LEFT JOIN csql ON csql.opportunity_id = gtmi.crm_opportunity_id
        LEFT JOIN part ON part.id = gtmi.crm_opportunity_id
        WHERE gtmi.date_label = 'today' AND gtmi.opportunity_is_commissionable = true
          AND (gtmi.product_arr_usd > 0 OR gtmi.product_booking_arr_usd > 0)
          {region_filter}
          {segment_filter}
    ),
    slippage_calc AS (
        SELECT curr.*, ios.initial_closedate,
            d_init.year_quarter as initial_fq, d_curr.year_quarter as current_fq,
            CASE WHEN d_init.year_quarter = '{quarter}' THEN 1 ELSE 0 END as was_expected_in_cq,
            CASE WHEN d_init.year_quarter = '{quarter}' AND (d_curr.year_quarter != '{quarter}' OR d_curr.year_quarter IS NULL) THEN 1 ELSE 0 END as has_slipped
        FROM current_state curr
        INNER JOIN initial_opp_stage ios ON curr.crm_opportunity_id = ios.crm_opportunity_id
        LEFT JOIN foundational.finance.dim_date d_init ON ios.initial_closedate = d_init.the_date
        LEFT JOIN foundational.finance.dim_date d_curr ON curr.current_closedate = d_curr.the_date
    )
    SELECT 
        '{label_region}' as "Region",
        '{quarter}' as "Quarter",
        '{label_segment}' as "Segment",

        ROUND(SUM(CASE WHEN is_CCaaS=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CCaaS=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "CCaaS: Slippage Rate (%)",
        SUM(CASE WHEN is_CCaaS=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "CCaaS: Slipped Deals (#)",
        SUM(CASE WHEN is_CCaaS=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "CCaaS: Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_CSQL=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CSQL=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "CSQL: Slippage Rate (%)",
        SUM(CASE WHEN is_CSQL=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "CSQL: Slipped Deals (#)",
        SUM(CASE WHEN is_CSQL=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "CSQL: Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_ES_Cross_Sell=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_ES_Cross_Sell=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "ES Cross-sell: Slippage Rate (%)",
        SUM(CASE WHEN is_ES_Cross_Sell=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "ES Cross-sell: Slipped Deals (#)",
        SUM(CASE WHEN is_ES_Cross_Sell=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "ES Cross-sell: Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_BDR=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_BDR=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "New Customer (BDR): Slippage Rate (%)",
        SUM(CASE WHEN is_New_Cust_BDR=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "New Customer (BDR): Slipped Deals (#)",
        SUM(CASE WHEN is_New_Cust_BDR=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "New Customer (BDR): Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_AE=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_AE=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "New Customer (AE): Slippage Rate (%)",
        SUM(CASE WHEN is_New_Cust_AE=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "New Customer (AE): Slipped Deals (#)",
        SUM(CASE WHEN is_New_Cust_AE=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "New Customer (AE): Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_Partner_Influenced=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Influenced=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Partner (Influenced): Slippage Rate (%)",
        SUM(CASE WHEN is_Partner_Influenced=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "Partner (Influenced): Slipped Deals (#)",
        SUM(CASE WHEN is_Partner_Influenced=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "Partner (Influenced): Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_Partner_Sourced=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Sourced=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Partner (Sourced): Slippage Rate (%)",
        SUM(CASE WHEN is_Partner_Sourced=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "Partner (Sourced): Slipped Deals (#)",
        SUM(CASE WHEN is_Partner_Sourced=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "Partner (Sourced): Initial Pipeline (#)",

        ROUND(SUM(CASE WHEN is_AI_group=1 AND has_slipped=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_AI_group=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END),0) * 100, 2) as "AI: Slippage Rate (%)",
        SUM(CASE WHEN is_AI_group=1 AND has_slipped=1 THEN 1 ELSE 0 END) as "AI: Slipped Deals (#)",
        SUM(CASE WHEN is_AI_group=1 AND was_expected_in_cq=1 THEN 1 ELSE 0 END) as "AI: Initial Pipeline (#)"
    FROM slippage_calc
    GROUP BY 1, 2, 3
    """
    return session.sql(query).to_pandas()


# ==========================================
# 3. S2 > S4 CONVERSION FUNCTION
# ==========================================
def get_conversion_rate_s2_to_s4_all_metrics(quarter, region=None, segment=None):
    
    region_filter = f"AND gtmi.region = '{region}'" if region else ""
    segment_filter = (
        f"AND gtmi.pro_forma_market_segment = '{segment}'" 
        if segment 
        else "AND gtmi.pro_forma_market_segment NOT IN ('SMB', 'Digital')"
    )
    label_region = region if region else "Global"
    label_segment = segment if segment else "Ent/Comm (Std)"

    query = f"""
    WITH initial_opp_stage AS (
        SELECT DISTINCT source_snapshot_date, crm_opportunity_id, opportunity_stage_name as initial_stage_name
        FROM foundational.customer.dim_crm_opportunities_daily_snapshot
        WHERE source_snapshot_date = (select dateadd(day, -28, max(source_snapshot_date)) from foundational.customer.dim_crm_opportunities_daily_snapshot)
    ),
    csql AS (SELECT id AS opportunity_id, campaign_id FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_BCV),
    part AS (SELECT id, partner, partner_deal_source FROM functional.gtm_sales_ops.partner_opp_table_all),
    current_state AS (
        SELECT gtmi.crm_opportunity_id, gtmi.stage_name as current_stage_name,
            CASE WHEN gtmi.PRODUCT IN ('Contact_Center') THEN 1 ELSE 0 END AS is_CCaaS,
            CASE WHEN csql.campaign_id LIKE '%70180000001JlouAAC%' THEN 1 ELSE 0 END AS is_CSQL,
            CASE WHEN (gtmi.PRODUCT IN ('ES') OR gtmi.use_case_c ILIKE '%internal%') AND gtmi.opportunity_type ILIKE '%Expansion%' THEN 1 ELSE 0 END AS is_ES_Cross_Sell,
            CASE WHEN gtmi.opportunity_type = 'New Business' THEN 1 ELSE 0 END AS is_New_Cust_Total,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%BDR%' THEN 1 ELSE 0 END AS is_New_Cust_BDR,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%AE Only%' THEN 1 ELSE 0 END AS is_New_Cust_AE,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Zendesk Sourced' THEN 1 ELSE 0 END AS is_Partner_Influenced,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Partner Sourced' THEN 1 ELSE 0 END AS is_Partner_Sourced,
            CASE WHEN gtmi.PRODUCT IN ('AI_Expert', 'AR', 'Copilot', 'Ultimate') THEN 1 ELSE 0 END AS is_AI_group
        FROM functional.gtm_sales_ops.gtmsi_consolidated_pipeline_bookings gtmi
        LEFT JOIN csql ON csql.opportunity_id = gtmi.crm_opportunity_id
        LEFT JOIN part ON part.id = gtmi.crm_opportunity_id
        WHERE gtmi.date_label = 'today' AND gtmi.opportunity_is_commissionable = true
          AND (gtmi.product_arr_usd > 0 OR gtmi.product_booking_arr_usd > 0)
          AND gtmi.close_year_quarter = '{quarter}'
          {region_filter}
          {segment_filter}
    ),
    conversion_calc AS (
        SELECT curr.*, ios.initial_stage_name,
            CASE WHEN ios.initial_stage_name = '02 - Confirm Need' THEN 1 ELSE 0 END AS was_S2,
            CASE WHEN ios.initial_stage_name = '02 - Confirm Need' AND curr.current_stage_name IN ('04 - Demonstrate Value', '05 - Secure Commitment', '06 - Contracting') THEN 1 ELSE 0 END AS has_converted_S2_S4
        FROM current_state curr
        INNER JOIN initial_opp_stage ios ON curr.crm_opportunity_id = ios.crm_opportunity_id
    )
    SELECT 
        '{label_region}' as "Region",
        '{quarter}' as "Quarter",
        '{label_segment}' as "Segment",
        
        ROUND(SUM(CASE WHEN is_CCaaS=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CCaaS=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "CCaaS: S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_CCaaS=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "CCaaS: Converted Deals (#)",
        SUM(CASE WHEN is_CCaaS=1 AND was_S2=1 THEN 1 ELSE 0 END) as "CCaaS: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_CSQL=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CSQL=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "CSQL: S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_CSQL=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "CSQL: Converted Deals (#)",
        SUM(CASE WHEN is_CSQL=1 AND was_S2=1 THEN 1 ELSE 0 END) as "CSQL: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_ES_Cross_Sell=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_ES_Cross_Sell=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "ES Cross-sell: S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_ES_Cross_Sell=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "ES Cross-sell: Converted Deals (#)",
        SUM(CASE WHEN is_ES_Cross_Sell=1 AND was_S2=1 THEN 1 ELSE 0 END) as "ES Cross-sell: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_Total=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_Total=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Total NB: S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_New_Cust_Total=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "Total NB: Converted Deals (#)",
        SUM(CASE WHEN is_New_Cust_Total=1 AND was_S2=1 THEN 1 ELSE 0 END) as "Total NB: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_BDR=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_BDR=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "New Customer (BDR): S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_New_Cust_BDR=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "New Customer (BDR): Converted Deals (#)",
        SUM(CASE WHEN is_New_Cust_BDR=1 AND was_S2=1 THEN 1 ELSE 0 END) as "New Customer (BDR): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_AE=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_AE=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "New Customer (AE): S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_New_Cust_AE=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "New Customer (AE): Converted Deals (#)",
        SUM(CASE WHEN is_New_Cust_AE=1 AND was_S2=1 THEN 1 ELSE 0 END) as "New Customer (AE): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_Partner_Influenced=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Influenced=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Partner (Influenced): S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_Partner_Influenced=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "Partner (Influenced): Converted Deals (#)",
        SUM(CASE WHEN is_Partner_Influenced=1 AND was_S2=1 THEN 1 ELSE 0 END) as "Partner (Influenced): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_Partner_Sourced=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Sourced=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Partner (Sourced): S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_Partner_Sourced=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "Partner (Sourced): Converted Deals (#)",
        SUM(CASE WHEN is_Partner_Sourced=1 AND was_S2=1 THEN 1 ELSE 0 END) as "Partner (Sourced): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_AI_group=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_AI_group=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "AI: S2 > S4 Conv Rate (%)",
        SUM(CASE WHEN is_AI_group=1 AND has_converted_S2_S4=1 THEN 1 ELSE 0 END) as "AI: Converted Deals (#)",
        SUM(CASE WHEN is_AI_group=1 AND was_S2=1 THEN 1 ELSE 0 END) as "AI: Initial S2 Cohort (#)"
    FROM conversion_calc
    GROUP BY 1, 2, 3
    """
    return session.sql(query).to_pandas()


# ==========================================
# 4. S2 > S3 CONVERSION FUNCTION
# ==========================================
def get_conversion_rate_s2_to_s3_all_metrics(quarter, region=None, segment=None):
    
    region_filter = f"AND gtmi.region = '{region}'" if region else ""
    segment_filter = (
        f"AND gtmi.pro_forma_market_segment = '{segment}'" 
        if segment 
        else "AND gtmi.pro_forma_market_segment NOT IN ('SMB', 'Digital')"
    )
    label_region = region if region else "Global"
    label_segment = segment if segment else "Ent/Comm (Std)"

    query = f"""
    WITH initial_opp_stage AS (
        SELECT DISTINCT source_snapshot_date, crm_opportunity_id, opportunity_stage_name as initial_stage_name
        FROM foundational.customer.dim_crm_opportunities_daily_snapshot
        WHERE source_snapshot_date = (select dateadd(day, -28, max(source_snapshot_date)) from foundational.customer.dim_crm_opportunities_daily_snapshot)
    ),
    csql AS (SELECT id AS opportunity_id, campaign_id FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_BCV),
    part AS (SELECT id, partner, partner_deal_source FROM functional.gtm_sales_ops.partner_opp_table_all),
    current_state AS (
        SELECT gtmi.crm_opportunity_id, gtmi.stage_name as current_stage_name,
            CASE WHEN gtmi.PRODUCT IN ('Contact_Center') THEN 1 ELSE 0 END AS is_CCaaS,
            CASE WHEN csql.campaign_id LIKE '%70180000001JlouAAC%' THEN 1 ELSE 0 END AS is_CSQL,
            CASE WHEN (gtmi.PRODUCT IN ('ES') OR gtmi.use_case_c ILIKE '%internal%') AND gtmi.opportunity_type ILIKE '%Expansion%' THEN 1 ELSE 0 END AS is_ES_Cross_Sell,
            CASE WHEN gtmi.opportunity_type = 'New Business' THEN 1 ELSE 0 END AS is_New_Cust_Total,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%BDR%' THEN 1 ELSE 0 END AS is_New_Cust_BDR,
            CASE WHEN gtmi.opportunity_type = 'New Business' AND gtmi.gtm_team LIKE '%AE Only%' THEN 1 ELSE 0 END AS is_New_Cust_AE,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Zendesk Sourced' THEN 1 ELSE 0 END AS is_Partner_Influenced,
            CASE WHEN part.partner IS NOT NULL AND part.partner != 'AWS Marketplace' AND part.partner_deal_source = 'Partner Sourced' THEN 1 ELSE 0 END AS is_Partner_Sourced,
            CASE WHEN gtmi.PRODUCT IN ('AI_Expert', 'AR', 'Copilot', 'Ultimate') THEN 1 ELSE 0 END AS is_AI_group
        FROM functional.gtm_sales_ops.gtmsi_consolidated_pipeline_bookings gtmi
        LEFT JOIN csql ON csql.opportunity_id = gtmi.crm_opportunity_id
        LEFT JOIN part ON part.id = gtmi.crm_opportunity_id
        WHERE gtmi.date_label = 'today' AND gtmi.opportunity_is_commissionable = true
          AND (gtmi.product_arr_usd > 0 OR gtmi.product_booking_arr_usd > 0)
          AND gtmi.close_year_quarter = '{quarter}'
          {region_filter}
          {segment_filter}
    ),
    conversion_calc AS (
        SELECT curr.*, ios.initial_stage_name,
            CASE WHEN ios.initial_stage_name = '02 - Confirm Need' THEN 1 ELSE 0 END AS was_S2,
            CASE WHEN ios.initial_stage_name = '02 - Confirm Need' AND curr.current_stage_name = '03 - Establish Value' THEN 1 ELSE 0 END AS has_converted_S2_S3
        FROM current_state curr
        INNER JOIN initial_opp_stage ios ON curr.crm_opportunity_id = ios.crm_opportunity_id
    )
    SELECT 
        '{label_region}' as "Region",
        '{quarter}' as "Quarter",
        '{label_segment}' as "Segment",

        ROUND(SUM(CASE WHEN is_CCaaS=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CCaaS=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "CCaaS: S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_CCaaS=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "CCaaS: Converted S3 Deals (#)",
        SUM(CASE WHEN is_CCaaS=1 AND was_S2=1 THEN 1 ELSE 0 END) as "CCaaS: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_CSQL=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_CSQL=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "CSQL: S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_CSQL=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "CSQL: Converted S3 Deals (#)",
        SUM(CASE WHEN is_CSQL=1 AND was_S2=1 THEN 1 ELSE 0 END) as "CSQL: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_ES_Cross_Sell=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_ES_Cross_Sell=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "ES Cross-sell: S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_ES_Cross_Sell=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "ES Cross-sell: Converted S3 Deals (#)",
        SUM(CASE WHEN is_ES_Cross_Sell=1 AND was_S2=1 THEN 1 ELSE 0 END) as "ES Cross-sell: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_Total=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_Total=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Total NB: S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_New_Cust_Total=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "Total NB: Converted S3 Deals (#)",
        SUM(CASE WHEN is_New_Cust_Total=1 AND was_S2=1 THEN 1 ELSE 0 END) as "Total NB: Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_BDR=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_BDR=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "New Customer (BDR): S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_New_Cust_BDR=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "New Customer (BDR): Converted S3 Deals (#)",
        SUM(CASE WHEN is_New_Cust_BDR=1 AND was_S2=1 THEN 1 ELSE 0 END) as "New Customer (BDR): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_New_Cust_AE=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_New_Cust_AE=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "New Customer (AE): S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_New_Cust_AE=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "New Customer (AE): Converted S3 Deals (#)",
        SUM(CASE WHEN is_New_Cust_AE=1 AND was_S2=1 THEN 1 ELSE 0 END) as "New Customer (AE): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_Partner_Influenced=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Influenced=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Partner (Influenced): S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_Partner_Influenced=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "Partner (Influenced): Converted S3 Deals (#)",
        SUM(CASE WHEN is_Partner_Influenced=1 AND was_S2=1 THEN 1 ELSE 0 END) as "Partner (Influenced): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_Partner_Sourced=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_Partner_Sourced=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "Partner (Sourced): S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_Partner_Sourced=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "Partner (Sourced): Converted S3 Deals (#)",
        SUM(CASE WHEN is_Partner_Sourced=1 AND was_S2=1 THEN 1 ELSE 0 END) as "Partner (Sourced): Initial S2 Cohort (#)",

        ROUND(SUM(CASE WHEN is_AI_group=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN is_AI_group=1 AND was_S2=1 THEN 1 ELSE 0 END),0) * 100, 2) as "AI: S2 > S3 Conv Rate (%)",
        SUM(CASE WHEN is_AI_group=1 AND has_converted_S2_S3=1 THEN 1 ELSE 0 END) as "AI: Converted S3 Deals (#)",
        SUM(CASE WHEN is_AI_group=1 AND was_S2=1 THEN 1 ELSE 0 END) as "AI: Initial S2 Cohort (#)"
    FROM conversion_calc
    GROUP BY 1, 2, 3
    """
    return session.sql(query).to_pandas()




In [ ]:
from functools import reduce
import pandas as pd

# Define the Quarter once
TARGET_QUARTER = '2025Q4'

# ==========================================
# 1. SEGMENTS (Digital & SMB)
# ==========================================

# --- Digital ---
df_win_rate_Digital = get_win_rate_data_all_metrics(quarter=TARGET_QUARTER, segment='Digital')
df_slippage_Digital = get_slippage_data_all_metrics(quarter=TARGET_QUARTER, segment='Digital')
df_conv_s2_s4_Digital = get_conversion_rate_s2_to_s4_all_metrics(quarter=TARGET_QUARTER, segment='Digital')
df_conv_s2_s3_Digital = get_conversion_rate_s2_to_s3_all_metrics(quarter=TARGET_QUARTER, segment='Digital')

# --- SMB ---
df_win_rate_SMB = get_win_rate_data_all_metrics(quarter=TARGET_QUARTER, segment='SMB')
df_slippage_SMB = get_slippage_data_all_metrics(quarter=TARGET_QUARTER, segment='SMB')
df_conv_s2_s4_SMB = get_conversion_rate_s2_to_s4_all_metrics(quarter=TARGET_QUARTER, segment='SMB')
df_conv_s2_s3_SMB = get_conversion_rate_s2_to_s3_all_metrics(quarter=TARGET_QUARTER, segment='SMB')


# ==========================================
# 2. REGIONS (NA, EMEA, LATAM, APAC)
# ==========================================

# --- NA ---
df_win_rate_NA = get_win_rate_data_all_metrics(quarter=TARGET_QUARTER, region='NA')
df_slippage_NA = get_slippage_data_all_metrics(quarter=TARGET_QUARTER, region='NA')
df_conv_s2_s4_NA = get_conversion_rate_s2_to_s4_all_metrics(quarter=TARGET_QUARTER, region='NA')
df_conv_s2_s3_NA = get_conversion_rate_s2_to_s3_all_metrics(quarter=TARGET_QUARTER, region='NA')

# --- EMEA ---
df_win_rate_EMEA = get_win_rate_data_all_metrics(quarter=TARGET_QUARTER, region='EMEA')
df_slippage_EMEA = get_slippage_data_all_metrics(quarter=TARGET_QUARTER, region='EMEA')
df_conv_s2_s4_EMEA = get_conversion_rate_s2_to_s4_all_metrics(quarter=TARGET_QUARTER, region='EMEA')
df_conv_s2_s3_EMEA = get_conversion_rate_s2_to_s3_all_metrics(quarter=TARGET_QUARTER, region='EMEA')

# --- LATAM ---
df_win_rate_LATAM = get_win_rate_data_all_metrics(quarter=TARGET_QUARTER, region='LATAM')
df_slippage_LATAM = get_slippage_data_all_metrics(quarter=TARGET_QUARTER, region='LATAM')
df_conv_s2_s4_LATAM = get_conversion_rate_s2_to_s4_all_metrics(quarter=TARGET_QUARTER, region='LATAM')
df_conv_s2_s3_LATAM = get_conversion_rate_s2_to_s3_all_metrics(quarter=TARGET_QUARTER, region='LATAM')

# --- APAC ---
df_win_rate_APAC = get_win_rate_data_all_metrics(quarter=TARGET_QUARTER, region='APAC')
df_slippage_APAC = get_slippage_data_all_metrics(quarter=TARGET_QUARTER, region='APAC')
df_conv_s2_s4_APAC = get_conversion_rate_s2_to_s4_all_metrics(quarter=TARGET_QUARTER, region='APAC')
df_conv_s2_s3_APAC = get_conversion_rate_s2_to_s3_all_metrics(quarter=TARGET_QUARTER, region='APAC')


# ==========================================
# ==========================================
# 3. CONSOLIDATE EVERYTHING (Updated)
# ==========================================

dfs_to_merge = [
    # Digital
    df_win_rate_Digital, df_slippage_Digital, df_conv_s2_s4_Digital, df_conv_s2_s3_Digital,
    # SMB
    df_win_rate_SMB, df_slippage_SMB, df_conv_s2_s4_SMB, df_conv_s2_s3_SMB,
    # NA
    df_win_rate_NA, df_slippage_NA, df_conv_s2_s4_NA, df_conv_s2_s3_NA,
    # EMEA
    df_win_rate_EMEA, df_slippage_EMEA, df_conv_s2_s4_EMEA, df_conv_s2_s3_EMEA,
    # LATAM
    df_win_rate_LATAM, df_slippage_LATAM, df_conv_s2_s4_LATAM, df_conv_s2_s3_LATAM,
    # APAC
    df_win_rate_APAC, df_slippage_APAC, df_conv_s2_s4_APAC, df_conv_s2_s3_APAC
]

# 1. Merge all into one Master DataFrame
master_df = reduce(lambda left, right: pd.merge(left, right, on=['Region', 'Quarter', 'Segment'], how='outer'), dfs_to_merge)

# 2. Create the 'Scenario' Column
# Logic: If Region is 'Global', use the Segment name (e.g., 'Digital'). 
#        Otherwise, use the Region name (e.g., 'NA').
master_df['Scenario'] = master_df.apply(
    lambda row: row['Segment'] if row['Region'] == 'Global' else row['Region'], 
    axis=1
)

# 3. Reorder Columns to put 'Scenario' first
cols = ['Scenario'] + [c for c in master_df.columns if c != 'Scenario']
master_df = master_df[cols]

# 4. Display Transposed (The 'Scenario' row will now be at the very top)
master_df